## **Projeto:** Merca Data Platform — Propensao a Cancelamento
### Feature Engineering v1 - Somente ecommerce_pedidos (squad3)
### Escopo desta versao
Usa apenas a tabela de cabecalho (ecommerce_pedidos). itens_pedido e enderecos ficam para uma proxima iteracao. Este notebook:
1. Deduplica (bug de reprocessamento 4x ja identificado)
2. Normaliza texto (status_pedido, metodo_pagamento)
3. Limpa / valida valores numericos
4. Deriva features de tempo e proporcao
5. Constroi historico do cliente com corte temporal (anti-leakage)
6. Codifica categoricos em numerico
7. Gera a tabela final pronta para o LightGBM
### Regra de ouro (anti-leakage)
dt_ultima_atualizacao_status e silver_processed_at NUNCA entram como feature.

In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
def get_squad3_client():
    return get_adls_client().get_file_system_client("squad3")
 
def ler_delta_silver_squad3(nome_tabela: str) -> "pyspark.sql.DataFrame":
    container_client = get_squad3_client()
    path_silver       = f"silver/{nome_tabela}"
    log.info(f"Lendo Silver (squad3): {nome_tabela}")
 
    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]
    if not paths:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {path_silver}")
    log.info(f"{len(paths)} arquivo(s) parquet encontrado(s)")
 
    BATCH_SIZE = 20
    df_final = None
    lote_pandas = []
    for i, file_path in enumerate(paths, start=1):
        file_client = container_client.get_file_client(file_path)
        data        = file_client.download_file().readall()
        lote_pandas.append(pd.read_parquet(io.BytesIO(data)))
        if len(lote_pandas) >= BATCH_SIZE or i == len(paths):
            df_lote_spark = spark.createDataFrame(pd.concat(lote_pandas, ignore_index=True))
            df_final = df_lote_spark if df_final is None else df_final.union(df_lote_spark)
            log.info(f"Progresso: {i}/{len(paths)} arquivos processados")
            lote_pandas = []
    return df_final

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, row_number, upper, trim, when, isnull,
    dayofweek, month, hour, year, round as spark_round
)
 
df_raw = ler_delta_silver_squad3("ecommerce_pedidos")
total_bruto = df_raw.count()
 
w = Window.partitionBy("id_pedido").orderBy(col("silver_processed_at").desc())
df = df_raw.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")
 
total_dedup = df.count()
log.info(f"Bruto: {total_bruto:,}  ->  Apos dedup: {total_dedup:,}  "
         f"(removidas {total_bruto - total_dedup:,} duplicatas)")

In [0]:
df = df \
    .withColumn("status_norm", upper(trim(col("status_pedido")))) \
    .withColumn("metodo_pagamento_norm", upper(trim(col("metodo_pagamento"))))
 
log.info("Valores distintos de metodo_pagamento_norm:")
df.groupBy("metodo_pagamento_norm").count().orderBy(col("count").desc()).show(truncate=False)

In [0]:
# Checa nulos e valores fora de faixa antes de decidir o que fazer com eles
qtd_nulos_valor      = df.filter(isnull(col("valor_total"))).count()
qtd_zero_ou_negativo = df.filter(col("valor_total") <= 0).count()
qtd_nulos_frete      = df.filter(isnull(col("valor_frete"))).count()
qtd_nulos_metodo     = df.filter(isnull(col("metodo_pagamento_norm"))).count()
qtd_nulos_cliente    = df.filter(isnull(col("id_cliente"))).count()
 
log.info(f"valor_total nulo            : {qtd_nulos_valor:,}")
log.info(f"valor_total <= 0            : {qtd_zero_ou_negativo:,}")
log.info(f"valor_frete nulo            : {qtd_nulos_frete:,}")
log.info(f"metodo_pagamento nulo       : {qtd_nulos_metodo:,}")
log.info(f"id_cliente nulo             : {qtd_nulos_cliente:,}")
 
# Remove linhas onde o valor do pedido - a informacao mais critica - e invalido.
# Nao imputamos valor_total (inventar um valor de venda seria arriscado demais);
# preferimos excluir esses casos raros e documentar a perda.
total_antes_limpeza = df.count()
df = df.filter(
    (~isnull(col("valor_total"))) &
    (col("valor_total") > 0) &
    (~isnull(col("id_cliente")))
)
total_apos_limpeza = df.count()
log.info(f"Linhas removidas na limpeza: {total_antes_limpeza - total_apos_limpeza:,}")
 
# valor_frete nulo e mais aceitavel (pode ser frete gratis mal registrado) -> tratamos como 0
df = df.withColumn("valor_frete", when(isnull(col("valor_frete")), 0).otherwise(col("valor_frete")))
 
# metodo_pagamento nulo vira categoria propria "DESCONHECIDO", em vez de descartar o pedido
df = df.withColumn(
    "metodo_pagamento_norm",
    when(isnull(col("metodo_pagamento_norm")), "DESCONHECIDO").otherwise(col("metodo_pagamento_norm"))
)

In [0]:
df = df \
    .withColumn("razao_frete_valor", spark_round(col("valor_frete") / col("valor_total"), 4)) \
    .withColumn("dia_semana_pedido", dayofweek(col("dt_pedido"))) \
    .withColumn("mes_pedido",        month(col("dt_pedido"))) \
    .withColumn("hora_pedido",       hour(col("dt_pedido"))) \
    .withColumn("ano_pedido",        year(col("dt_pedido")))  # mantido so para referencia/split, nao e feature de entrada
 
log.info("Amostra das features derivadas:")
df.select("id_pedido", "valor_total", "valor_frete", "razao_frete_valor",
          "dia_semana_pedido", "mes_pedido", "hora_pedido").show(5, truncate=False)

In [0]:
# Self-join: para cada pedido, conta APENAS pedidos do mesmo cliente com
# dt_pedido ANTERIOR ao pedido atual. Isso impede que o modelo "veja o futuro".
df_base = df.select("id_pedido", "id_cliente", "dt_pedido", "status_norm").alias("atual")
df_hist = df.select("id_cliente", "dt_pedido", "status_norm").alias("historico")
 
df_join = df_base.join(
    df_hist,
    (col("atual.id_cliente") == col("historico.id_cliente")) &
    (col("historico.dt_pedido") < col("atual.dt_pedido")),
    how="left"
)
 
from pyspark.sql.functions import count as spark_count, sum as spark_sum, coalesce, lit
 
df_historico_cliente = df_join.groupBy(col("atual.id_pedido").alias("id_pedido")).agg(
    spark_count(col("historico.dt_pedido")).alias("qtd_pedidos_anteriores"),
    spark_sum(
        when(col("historico.status_norm") == "CANCELADO", 1).otherwise(0)
    ).alias("qtd_cancelamentos_anteriores")
)
 
df_historico_cliente = df_historico_cliente \
    .withColumn("qtd_cancelamentos_anteriores", coalesce(col("qtd_cancelamentos_anteriores"), lit(0))) \
    .withColumn(
        "taxa_cancelamento_historica",
        when(col("qtd_pedidos_anteriores") > 0,
             spark_round(col("qtd_cancelamentos_anteriores") / col("qtd_pedidos_anteriores"), 4))
        .otherwise(lit(0.0))
    )
 
df = df.join(df_historico_cliente, on="id_pedido", how="left")
 
log.info("Distribuicao de qtd_pedidos_anteriores (cliente novo vs recorrente):")
df.groupBy(
    when(col("qtd_pedidos_anteriores") == 0, "cliente novo (0 pedidos antes)")
    .otherwise("cliente recorrente")
    .alias("tipo_cliente")
).count().show(truncate=False)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number
 
df_categorias = df.select("metodo_pagamento_norm").distinct()
w_cat = Window.orderBy("metodo_pagamento_norm")
df_categorias = df_categorias.withColumn(
    "metodo_pagamento_idx",
    row_number().over(w_cat) - 1  # comeca em 0
)
 
df = df.join(df_categorias, on="metodo_pagamento_norm", how="left")
 
log.info("Mapeamento metodo_pagamento -> indice numerico:")
df.select("metodo_pagamento_norm", "metodo_pagamento_idx").distinct() \
  .orderBy("metodo_pagamento_idx").show(truncate=False)

In [0]:
df_final = df \
    .withColumn(
        "target_cancelamento",
        when(col("status_norm") == "CANCELADO", 1).otherwise(0)
    ) \
    .select(
        "id_pedido",
        "id_cliente",
        "dt_pedido",            # mantido so para split temporal, nao usar como feature direta
        "ano_pedido",           # idem
        "valor_total",
        "valor_frete",
        "razao_frete_valor",
        "metodo_pagamento_norm",
        "metodo_pagamento_idx",
        "dia_semana_pedido",
        "mes_pedido",
        "hora_pedido",
        "qtd_pedidos_anteriores",
        "qtd_cancelamentos_anteriores",
        "taxa_cancelamento_historica",
        "target_cancelamento"
    )

In [0]:
total_final = df_final.count()
positivos   = df_final.filter(col("target_cancelamento") == 1).count()
 
log.info(f"Total de linhas na tabela final : {total_final:,}")
log.info(f"Positivos (target=1)            : {positivos:,}  ({round(100*positivos/total_final,2)}%)")
log.info(f"Negativos (target=0)            : {total_final - positivos:,}")
 
log.info("Checagem de nulos por coluna (deve ser tudo zero, exceto colunas propositalmente nulaveis):")
for c in df_final.columns:
    qtd_nulos = df_final.filter(isnull(col(c))).count()
    if qtd_nulos > 0:
        log.warning(f"  {c}: {qtd_nulos:,} nulos")
 
df_final.printSchema()
df_final.show(5, truncate=False)

In [0]:
CAMADA_FEATURES = "ml_features"
TABELA_FEATURES = "propensao_cancelamento_pedidos_v1"
 
sucesso_gravacao = gravar_delta(df_final, CAMADA_FEATURES, TABELA_FEATURES, mode="overwrite")
 
if sucesso_gravacao:
    log.info(f"Tabela de features gravada em: {get_delta_path(CAMADA_FEATURES, TABELA_FEATURES)}")
else:
    log.error("Falha ao gravar a tabela de features - notebook de treino nao tera de onde ler.")